<a href="https://colab.research.google.com/github/Shaan0104/CSC580_HighwayEnv_RL_Experiment/blob/main/CSC_580_RL_EXPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CSC 580 - ARTIFICIAL INTELLIGENCE 2

SYEDA AFSHAAN FATHIMA

Final Project: option 2

#project overview
This notebook presents the extensive experiments on the "highway-v0" environment from gymnasium. using 3 rl algorithms:
DQN, PPO AND A2C, implemented via stable baseline3.
The Goal is to derieve optimal hyperparameters and compare algorithm performance across multiple evaluation metrics.



# CSC 580 – Artificial Intelligence II | Winter 2026
## Final Project – Option 2: Highway-Env RL Experiments
---
**Name:** Syeda Afshaan Fathima
**Course/Section:** CSC 580 – Winter 2026
**Assignment:** Final Project – Option 2
**AI Tools Consulted:** Claude (Anthropic)
---
## Project Overview
This notebook presents extensive experimentation on the `highway-v0`
environment from Gymnasium using three reinforcement learning algorithms:
**DQN**, **PPO**, and **A2C**, implemented via Stable-Baselines3.
The goal is to derive optimal hyperparameters and compare algorithm
performance across multiple evaluation metrics.

In [1]:
#installs required libraries
!pip install highway-env
!pip install stable-baselines3[extra]
!pip install imageio imageio-ffmpeg pygame
import warnings
warnings.filterwarnings("ignore")

print(f"All libraries have been sucessfully installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 6.9 MB/s eta 0:00:00
All libraries have been sucessfully installed!


In [2]:
import gymnasium as gym
import highway_env
from stable_baselines3 import DQN, PPO, A2C
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor


#importing matplot and data libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

#OS and file management
import os
import warnings
warnings.filterwarnings("ignore")

print(f"Successfully imported all libraries!")

Successfully imported all libraries!


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
#Creating the highway-v0 environment and disabling rendering for training
env = gym.make("highway-v0", render_mode =None)

#Environment reset to get the initial state
obs, info = env.reset()

#environment details
print("\Highway-V0 Environment Overview\n")

#what the agent sees at each timestep is observation space
print(f"\nObservation Space: {env.observation_space}")
print(f"Shape: {env.observation_space.shape}")

#what actions the agent can take is Action space
print(f"Action Space: {env.action_space}")
print(f" Total Actions: {env.action_space.n}")
print("""Actions Available:
   0 = LANE_LEFT (move left)
   1 = IDLE (stay in lane)
   2 = LANE_RIGHT (move right)
   3 = FASTER (accelerate)
   4 = SLOWER (decelerate)""")

#Initial observation shape
print(f"Initial Observation Shape: {obs.shape}")
print(f"\nEnvironment initialized successfully!")

#closing environment after exploration
env.close()

\Highway-V0 Environment Overview


Observation Space: Box(-inf, inf, (5, 5), float32)
Shape: (5, 5)
Action Space: Discrete(5)
 Total Actions: 5
Actions Available:
   0 = LANE_LEFT (move left)
   1 = IDLE (stay in lane)
   2 = LANE_RIGHT (move right)
   3 = FASTER (accelerate)
   4 = SLOWER (decelerate)
Initial Observation Shape: (5, 5)

Environment initialized successfully!


#configuring environment
Here shared enviroinment configuration and hyperparameters used in all of the 3 models are defined to ensure fair comparison.

In [4]:
ENV_CONFIG = {
    "observation": {
        "type": "Kinematics",        #Used numerical vehicle data (not pixels)
        "vehicles_count": 5,         #it observe 5 nearby vehicles
        "features": ["presence", "x", "y", "vx", "vy"],  #and 5 features per vehicle
        "normalize": True,           #Normalizing values for stable training
    },
    "action": {
        "type": "DiscreteMetaAction" #using the 5 discrete actions
    },
    "lanes_count": 4,                #4 lane highway
    "vehicles_count": 20,            #20 other vehicles on the road
    "duration": 40,                  #Each episode lasts for 40 seconds
    "reward_speed_range": [20, 30],  #Reward agent for driving at 20-30 m/s
    "simulation_frequency": 15,      #Update rate
    "policy_frequency": 1,           #Agent makes decision every 1 step
}

#Training timesteps for each model
#30,000 steps for balanced training quality
TIMESTEPS = 30000

#Number of evaluation episodes after training
EVAL_EPISODES = 10

#Setting random seed for reproducibility across all experiments
SEED = 42

print("\nEXPERIMENT CONFIGURATION\n")
print(f"\nLanes: {ENV_CONFIG['lanes_count']}")
print(f"Other Vehicles: {ENV_CONFIG['vehicles_count']}")
print(f"Episode Duration: {ENV_CONFIG['duration']} seconds")
print(f"Training Timesteps: {TIMESTEPS:,}")
print(f"Evaluation Episodes: {EVAL_EPISODES}")
print(f"Random Seed: {SEED}")
print(f"\nSuccessfully Configurared!")


EXPERIMENT CONFIGURATION


Lanes: 4
Other Vehicles: 20
Episode Duration: 40 seconds
Training Timesteps: 30,000
Evaluation Episodes: 10
Random Seed: 42

Successfully Configurared!


## Section 3: Model 1 – Deep Q-Network (DQN)
DQN is a value-based reinforcement learning algorithm that uses a neural
network to approximate the Q-value function. It learns by estimating the
expected cumulative reward for each state-action pair and selects the action
with the highest Q-value. DQN introduced Experience Replay and a Target
Network to stabilize training — both critical innovations in Deep RL.

In [ ]:
# ============================================================
# Cell 5: Train Model 1 – DQN (Deep Q-Network)
# ============================================================

# Create the highway environment for DQN training
dqn_env = gym.make("highway-v0", render_mode=None)

# Apply our shared configuration to the environment
dqn_env.unwrapped.configure(ENV_CONFIG)

# Wrap with Monitor to automatically track rewards and episode lengths
dqn_env = Monitor(dqn_env)

# Reset environment with fixed seed for reproducibility
dqn_env.reset(seed=SEED)

# Initialize DQN model with MlpPolicy (Multi-Layer Perceptron)
# MlpPolicy is appropriate here since our observations are numerical vectors
dqn_model = DQN(
    policy="MlpPolicy",
    env=dqn_env,
    learning_rate=5e-4,        # How fast the network updates its weights
    batch_size=64,             # Number of experiences sampled per update
    buffer_size=15000,         # Size of the experience replay buffer
    learning_starts=200,       # Steps before learning begins
    gamma=0.99,                # Discount factor: values future rewards highly
    train_freq=1,              # Train the network every step
    target_update_interval=50, # Steps between target network updates
    exploration_fraction=0.3,  # Fraction of training spent exploring
    exploration_final_eps=0.05,# Final exploration rate (5%)
    verbose=0,                 # Suppress training logs for clean output
    seed=SEED
)

print("=" * 50)
print("TRAINING MODEL 1: DQN")
print("=" * 50)
print(f"\n📋 Policy:            MlpPolicy")
print(f"📈 Learning Rate:     5e-4")
print(f"🗂️  Buffer Size:       15,000")
print(f"🎲 Exploration:       30% of training")
print(f"🔁 Timesteps:         {TIMESTEPS:,}")
print(f"\n⏳ Training started...")

# Train the DQN model
dqn_model.learn(total_timesteps=TIMESTEPS)

print(f"\n✅ DQN Training Complete!")

# Save the trained model for later evaluation and video generation
dqn_model.save("dqn_highway")
print("💾 DQN model saved as 'dqn_highway'")

# Close the training environment
dqn_env.close()

TRAINING MODEL 1: DQN

📋 Policy:            MlpPolicy
📈 Learning Rate:     5e-4
🗂️  Buffer Size:       15,000
🎲 Exploration:       30% of training
🔁 Timesteps:         30,000

⏳ Training started...


In [5]:
# ============================================================
# Cell 6: Evaluate DQN Model
# ============================================================

# Create a fresh environment for evaluation (separate from training env)
dqn_eval_env = gym.make("highway-v0", render_mode=None)
dqn_eval_env.unwrapped.configure(ENV_CONFIG)
dqn_eval_env = Monitor(dqn_eval_env)
dqn_eval_env.reset(seed=SEED)

# Load the saved DQN model
dqn_model = DQN.load("dqn_highway", env=dqn_eval_env)

# Evaluate the model over 10 episodes
# Returns mean reward and standard deviation
dqn_mean_reward, dqn_std_reward = evaluate_policy(
    dqn_model,
    dqn_eval_env,
    n_eval_episodes=EVAL_EPISODES,
    deterministic=True  # Use greedy policy during evaluation
)

print("=" * 50)
print("DQN EVALUATION RESULTS")
print("=" * 50)
print(f"\n🏆 Mean Reward:  {dqn_mean_reward:.2f}")
print(f"📊 Std Dev:      {dqn_std_reward:.2f}")
print(f"📈 Episodes:     {EVAL_EPISODES}")
print(f"\n✅ DQN Evaluation Complete!")

dqn_eval_env.close()

FileNotFoundError: [Errno 2] No such file or directory: 'dqn_highway.zip'

## Section 4: Model 2 – Proximal Policy Optimization (PPO)
PPO is a policy-gradient based reinforcement learning algorithm. Unlike DQN
which learns a value function, PPO directly optimizes the policy itself.
It uses a clipped surrogate objective to prevent overly large policy updates,
making it more stable than earlier policy gradient methods. PPO is one of
the most widely used RL algorithms due to its balance of simplicity,
stability, and performance.

In [8]:
# ============================================================
# Cell 7: Train Model 2 – PPO (Proximal Policy Optimization)
# ============================================================

# Create a fresh environment for PPO training
ppo_env = gym.make("highway-v0", render_mode=None)
ppo_env.unwrapped.configure(ENV_CONFIG)
ppo_env = Monitor(ppo_env)
ppo_env.reset(seed=SEED)

# Initialize PPO model
# PPO is an on-policy algorithm — it learns from current experience only
ppo_model = PPO(
    policy="MlpPolicy",
    env=ppo_env,
    learning_rate=5e-4,      # Step size for policy network updates
    n_steps=256,             # Steps collected before each policy update
    batch_size=64,           # Mini-batch size for gradient updates
    n_epochs=10,             # Number of passes over collected data
    gamma=0.99,              # Discount factor for future rewards
    gae_lambda=0.95,         # GAE parameter for advantage estimation
    clip_range=0.2,          # Clipping range to limit policy update size
    verbose=0,
    seed=SEED
)

print("=" * 50)
print("TRAINING MODEL 2: PPO")
print("=" * 50)
print(f"\n📋 Policy:            MlpPolicy")
print(f"📈 Learning Rate:     5e-4")
print(f"📦 N Steps:           256")
print(f"✂️  Clip Range:        0.2")
print(f"🔁 Timesteps:         {TIMESTEPS:,}")
print(f"\n⏳ Training started...")

# Train the PPO model
ppo_model.learn(total_timesteps=TIMESTEPS)

print(f"\n✅ PPO Training Complete!")

# Save the trained PPO model
ppo_model.save("ppo_highway")
print("💾 PPO model saved as 'ppo_highway'")

ppo_env.close()

TRAINING MODEL 2: PPO

📋 Policy:            MlpPolicy
📈 Learning Rate:     5e-4
📦 N Steps:           256
✂️  Clip Range:        0.2
🔁 Timesteps:         30,000

⏳ Training started...

✅ PPO Training Complete!
💾 PPO model saved as 'ppo_highway'


In [9]:
# ============================================================
# Cell 8: Evaluate PPO Model
# ============================================================

# Create a fresh environment for PPO evaluation
ppo_eval_env = gym.make("highway-v0", render_mode=None)
ppo_eval_env.unwrapped.configure(ENV_CONFIG)
ppo_eval_env = Monitor(ppo_eval_env)
ppo_eval_env.reset(seed=SEED)

# Load the saved PPO model
ppo_model = PPO.load("ppo_highway", env=ppo_eval_env)

# Evaluate PPO over 10 episodes
ppo_mean_reward, ppo_std_reward = evaluate_policy(
    ppo_model,
    ppo_eval_env,
    n_eval_episodes=EVAL_EPISODES,
    deterministic=True
)

print("=" * 50)
print("PPO EVALUATION RESULTS")
print("=" * 50)
print(f"\n🏆 Mean Reward:  {ppo_mean_reward:.2f}")
print(f"📊 Std Dev:      {ppo_std_reward:.2f}")
print(f"📈 Episodes:     {EVAL_EPISODES}")
print(f"\n✅ PPO Evaluation Complete!")

ppo_eval_env.close()

PPO EVALUATION RESULTS

🏆 Mean Reward:  29.16
📊 Std Dev:      0.51
📈 Episodes:     10

✅ PPO Evaluation Complete!


## Section 5: Model 3 – Advantage Actor-Critic (A2C)
A2C is a synchronous policy-gradient algorithm that combines two components:
an Actor that decides which action to take, and a Critic that evaluates
how good that action was using the Advantage function. Unlike PPO, A2C
updates the policy immediately after each step without clipping, making
it faster but potentially less stable. A2C is the synchronous version of
A3C (Asynchronous Advantage Actor-Critic) and is well suited for
environments with discrete action spaces like highway-v0.

In [10]:
# ============================================================
# Cell 9: Train Model 3 – A2C (Advantage Actor-Critic)
# ============================================================

# Create a fresh environment for A2C training
a2c_env = gym.make("highway-v0", render_mode=None)
a2c_env.unwrapped.configure(ENV_CONFIG)
a2c_env = Monitor(a2c_env)
a2c_env.reset(seed=SEED)

# Initialize A2C model
# A2C combines actor (policy) and critic (value) networks
a2c_model = A2C(
    policy="MlpPolicy",
    env=a2c_env,
    learning_rate=7e-4,      # Slightly higher LR than DQN/PPO — A2C converges faster
    n_steps=5,               # Steps before each update — A2C updates very frequently
    gamma=0.99,              # Discount factor for future rewards
    gae_lambda=1.0,          # GAE lambda — 1.0 means no GAE smoothing
    ent_coef=0.01,           # Entropy coefficient to encourage exploration
    vf_coef=0.5,             # Value function loss weight in total loss
    max_grad_norm=0.5,       # Gradient clipping for training stability
    verbose=0,
    seed=SEED
)

print("=" * 50)
print("TRAINING MODEL 3: A2C")
print("=" * 50)
print(f"\n📋 Policy:            MlpPolicy")
print(f"📈 Learning Rate:     7e-4")
print(f"📦 N Steps:           5")
print(f"🎯 Entropy Coef:      0.01")
print(f"🔁 Timesteps:         {TIMESTEPS:,}")
print(f"\n⏳ Training started...")

# Train the A2C model
a2c_model.learn(total_timesteps=TIMESTEPS)

print(f"\n✅ A2C Training Complete!")

# Save the trained A2C model
a2c_model.save("a2c_highway")
print("💾 A2C model saved as 'a2c_highway'")

a2c_env.close()

TRAINING MODEL 3: A2C

📋 Policy:            MlpPolicy
📈 Learning Rate:     7e-4
📦 N Steps:           5
🎯 Entropy Coef:      0.01
🔁 Timesteps:         30,000

⏳ Training started...

✅ A2C Training Complete!
💾 A2C model saved as 'a2c_highway'
